In [1]:
!pip install xgboost
!pip install lightgbm
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


In [8]:
# BREAST Cancer with ONLY researched biomarkers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Load
df = pd.read_csv("breast_cancer_extended.csv")

# ── Core breast biomarkers only ──────────────────────────────
core_features = [
   'Age',
'Smoking',
'Gender',
'BMI'
]

target_col = "Risk_Level"

X  = df[core_features]
le = LabelEncoder().fit(df[target_col])
y  = le.transform(df[target_col])

print(f"Features: {core_features}")
print(f"Shape: {X.shape}")
print(df[target_col].value_counts())

# Split + Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, max_depth=6,
                                                   min_samples_leaf=10,
                                                   class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42,
                                                   max_depth=10, min_samples_leaf=5,
                                                   class_weight='balanced'),
    "Extra Trees":         ExtraTreesClassifier(n_estimators=200, random_state=42,
                                                max_depth=10, min_samples_leaf=5,
                                                class_weight='balanced'),
    "AdaBoost":            AdaBoostClassifier(n_estimators=150, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=150, random_state=42,
                                                       max_depth=4, learning_rate=0.05,
                                                       subsample=0.8),
    "SVM (RBF Kernel)":    SVC(kernel='rbf', probability=True, random_state=42,
                               class_weight='balanced'),
    "XGBoost":             XGBClassifier(n_estimators=150, random_state=42,
                                          max_depth=4, learning_rate=0.05,
                                          subsample=0.8, reg_alpha=0.1,
                                          reg_lambda=1.0, eval_metric='logloss'),

    "CatBoost":            CatBoostClassifier(verbose=0, random_state=42,
                                               depth=4, learning_rate=0.05,
                                               l2_leaf_reg=5.0)
}

'''

    "LightGBM":            LGBMClassifier(n_estimators=150, random_state=42,
                                           max_depth=4, learning_rate=0.05,
                                           reg_alpha=0.5, reg_lambda=2.0,
                                           min_child_samples=30,
                                           class_weight='balanced'),
'''

# Train and evaluate
results     = {}
predictions = {}

print("\n=== Results: Core breast Features Only ===")
print(f"{'Model':<25} {'Test Acc':>10} {'Train Acc':>10} {'Gap':>8} {'Status':>10}")
print("-" * 68)

for name, model in models.items():
    if name == "XGBoost":
        sw = compute_sample_weight(class_weight='balanced', y=y_train)
        model.fit(X_train, y_train, sample_weight=sw)
    else:
        model.fit(X_train, y_train)

    y_pred    = model.predict(X_test)
    test_acc  = accuracy_score(y_test, y_pred)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    gap       = train_acc - test_acc
    status    = "⚠️ Overfit" if gap > 0.15 else "⚡ Slight" if gap > 0.05 else "✓ Good"

    results[name]     = test_acc
    predictions[name] = y_pred
    print(f"{name:<25} {test_acc:>10.4f} {train_acc:>10.4f} {gap:>8.4f} {status:>10}")

# Best model report
results_df      = pd.DataFrame(list(results.items()),
                                columns=["Model", "Accuracy"]).sort_values(
                                by="Accuracy", ascending=False)
best_model_name = results_df.iloc[0]['Model']

print(f"\n🏁 Best Model: {best_model_name} — {results_df.iloc[0]['Accuracy']:.4f}")
print(f"\n=== Classification Report: {best_model_name} ===")
print(classification_report(y_test, predictions[best_model_name],
                             target_names=le.classes_))

Features: ['Age', 'Smoking', 'Gender', 'BMI']
Shape: (4173, 4)
Risk_Level
Medium    3396
Low        636
High       141
Name: count, dtype: int64

=== Results: Core breast Features Only ===
Model                       Test Acc  Train Acc      Gap     Status
--------------------------------------------------------------------
Logistic Regression           0.2635     0.2645   0.0011     ✓ Good
Decision Tree                 0.3210     0.3385   0.0176     ✓ Good
Random Forest                 0.6754     0.8358   0.1604 ⚠️ Overfit
Extra Trees                   0.5030     0.6061   0.1031   ⚡ Slight
AdaBoost                      0.8012     0.8170   0.0158     ✓ Good
Gradient Boosting             0.7964     0.8277   0.0313     ✓ Good
SVM (RBF Kernel)              0.3653     0.4038   0.0386     ✓ Good
XGBoost                       0.4886     0.5920   0.1033   ⚡ Slight
CatBoost                      0.7964     0.8206   0.0241     ✓ Good

🏁 Best Model: AdaBoost — 0.8012

=== Classification Report: A

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
